In [1]:
!pip install selenium undetected-chromedriver "setuptools<70.0.0" numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 6.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: setuptools
    Found existing installation: setuptools 73.0.1
    Uninstalling setuptools-73.0.1:
      Successfully uninstalled setuptools-73.0.1


In [2]:
!pip install packaging

In [3]:
import sys
import types

class MinimalLooseVersion:
    def __init__(self, vstring):
        self.vstring = str(vstring)
        self.version = [int(x) if x.isdigit() else x for x in self.vstring.split('.')]

    def __lt__(self, other):
        return self.version < other.version
    def __gt__(self, other):
        return self.version > other.version
    def __eq__(self, other):
        return self.version == other.version
    def __ge__(self, other):
        return self.version >= other.version
    def __le__(self, other):
        return self.version <= other.version

# Khởi tạo mô-đun giả lập
distutils = types.ModuleType('distutils')
distutils.version = types.ModuleType('distutils.version')
distutils.version.LooseVersion = MinimalLooseVersion
sys.modules['distutils'] = distutils
sys.modules['distutils.version'] = distutils.version

# Import các thư viện gốc
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
import selenium.webdriver.support.expected_conditions as EC
import undetected_chromedriver as uc
import time
import numpy as np
import pandas as pd

In [4]:
options = uc.ChromeOptions()
# KHÔNG dùng --headless với Cloudflare
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

# Giả lập người dùng thật
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                     "AppleWebKit/537.36 (KHTML, like Gecko) "
                     "Chrome/120.0.0.0 Safari/537.36")

driver = uc.Chrome(options=options)

In [5]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import time

def wait_for_cloudflare(driver, timeout=30):
    """
    Hàm chờ trình duyệt vượt qua màn hình bảo vệ của Cloudflare.
    """
    # print("⏳ Đang kiểm tra Cloudflare...")
    try:
        # Cách phổ biến nhất: Chờ cho đến khi Title của trang KHÔNG CÒN là "Just a moment..."
        # (Lưu ý: Nếu web tiếng Việt, Cloudflare có thể hiển thị "Vui lòng chờ...")
        WebDriverWait(driver, timeout).until_not(
            EC.title_contains("Just a moment...")
        )
        
        # Đợi thêm 1-2 giây để các script của trang chính thức tải xong (chống lỗi Stale Element)
        time.sleep(2) 
        return True
    
    except Exception as e:
        print(f"❌ Vượt Cloudflare thất bại hoặc quá thời gian {timeout}s.")
        return False

In [6]:
def get_hrefs_from_page(url):
    driver.get(url)
    
    if not wait_for_cloudflare(driver, timeout=30):
        return []
    
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CLASS_NAME, "js__product-link-for-product-id"))
        )
    except:
        print(f"  ⚠️ Không tìm thấy element. Title: {driver.title}")
        return []
    
    elements = driver.find_elements(By.CLASS_NAME, "js__product-link-for-product-id")
    return [element.get_attribute("href") for element in elements]

base_path      = "https://batdongsan.com.vn/ban-nha-dat-tp-hcm"
query          = "?vrs=1"
all_hrefs      = []
page           = 1
TARGET_SAMPLES = 3500  # ⬅️ Đổi số này để kiểm soát đầu ra

while len(all_hrefs) < TARGET_SAMPLES:
    url = f"{base_path}{f'/p{page}' if page > 1 else ''}{query}"
    print(f"📄 Trang {page} | Đã có {len(all_hrefs)}/{TARGET_SAMPLES} tin: {url}")

    hrefs = get_hrefs_from_page(url)

    remaining = TARGET_SAMPLES - len(all_hrefs)
    all_hrefs.extend(hrefs[:remaining])  # Cắt đúng số còn thiếu

    next_page = driver.find_elements(
        By.CSS_SELECTOR, 'a.re__pagination-icon:not(.re__pagination-icon--no-effect)'
    )
    if not next_page:
        print("🏁 Đã hết trang.")
        break

    page += 1
    time.sleep(2)

print(f"\n✅ Hoàn thành: {len(all_hrefs)} tin từ {page} trang")
for href in all_hrefs:
    print(href)

driver.quit()

📄 Trang 1 | Đã có 0/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm?vrs=1
📄 Trang 2 | Đã có 31/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p2?vrs=1
📄 Trang 3 | Đã có 62/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p3?vrs=1
📄 Trang 4 | Đã có 93/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p4?vrs=1
📄 Trang 5 | Đã có 124/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p5?vrs=1
📄 Trang 6 | Đã có 155/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p6?vrs=1
📄 Trang 7 | Đã có 176/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p7?vrs=1
📄 Trang 8 | Đã có 207/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p8?vrs=1
📄 Trang 9 | Đã có 238/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p9?vrs=1
📄 Trang 10 | Đã có 269/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p10?vrs=1
📄 Trang 11 | Đã có 300/3500 tin: https://batdongsan.com.vn/ban-nha-dat-tp-hcm/p11?vrs=1
📄 Trang 12 | Đã có 331/3500 tin: https://batdongsan.com.vn

In [7]:
with open("hrefs.txt", "w", encoding="utf-8") as file:
    for href in all_hrefs:
        file.write(href + "\n")

print("Đã lưu tất cả href vào file hrefs.txt")

Đã lưu tất cả href vào file hrefs.txt


In [ ]:
def extract_property_info(url):
    """
    Cào thông tin chi tiết một bất động sản.
    Cấu trúc HTML target:
      - Địa chỉ : <span class='re__address-line-1'>
      - Thông số: <div class='re__pr-other-info-display'> > .re__pr-specs-content-item
    """
    # ── Khởi tạo dict với đúng key khớp title HTML ──────────────────────────────
    info = {
        'Link'                    : url,
        'Địa chỉ'                 : np.nan,
        'Khoảng giá'              : np.nan,   # title HTML: 'Khoảng giá'
        'Diện tích'               : np.nan,   # title HTML: 'Diện tích'
        'Số phòng ngủ'            : np.nan,   # title HTML: 'Số phòng ngủ'
        'Số phòng tắm, vệ sinh'   : np.nan,   # title HTML: 'Số phòng tắm, vệ sinh' (có dấu phẩy!)
        'Số tầng'                 : np.nan,   # title HTML: 'Số tầng'
        'Hướng nhà'               : np.nan,   # title HTML: 'Hướng nhà'
        'Hướng ban công'          : np.nan,   # title HTML: 'Hướng ban công'
        'Đường vào'               : np.nan,   # title HTML: 'Đường vào'
        'Pháp lý'                 : np.nan,   # title HTML: 'Pháp lý'
        'Nội thất'                : np.nan,   # title HTML: 'Nội thất'
        'Latitude'                : np.nan,
        'Longitude'               : np.nan,
    }

    try:
        driver.get(url)
        # Chờ khối thông số chính xuất hiện (tối đa 10s)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, 're__pr-other-info-display'))
        )
    except Exception:
        # ── 4. Mô tả (description) ───────────────────────────────────────────────────
    try:
        info['Mô tả'] = driver.find_element(
            By.CLASS_NAME, 're__section-body.re__detail-content'
        ).text.strip()
    except Exception:
        pass

    return info  # Trang lỗi / không tải được → trả về dict rỗng

    # ── 1. Địa chỉ: class MỚI là re__address-line-1 ─────────────────────────────
    # LỖI CŨ: dùng 're__pr-short-description' — class này không còn tồn tại
    try:
        info['Địa chỉ'] = driver.find_element(
            By.CLASS_NAME, 're__address-line-1'
        ).text.strip()
    except Exception:
        pass

    # ── 2. Tọa độ từ iframe bản đồ ───────────────────────────────────────────────
    try:
        data_src = driver.find_element(
            By.CSS_SELECTOR, 'iframe.lazyload'
        ).get_attribute('data-src')
        lat, lon = data_src.split('q=')[1].split('&')[0].split(',')
        info['Latitude']  = lat.strip()
        info['Longitude'] = lon.strip()
    except Exception:
        pass

    # ── 3. Các thông số trong re__pr-other-info-display ──────────────────────────
    # LỖI CŨ 1: selector '.re__pr-specs-content-item' lấy toàn trang → nhầm section
    # LỖI CŨ 2: key 'Số toilet' không khớp title HTML 'Số phòng tắm, vệ sinh' → NaN
    # FIX: scope selector vào đúng khối + key trong dict phải khớp chính xác title
    try:
        specs = driver.find_elements(
            By.CSS_SELECTOR,
            '.re__pr-other-info-display .re__pr-specs-content-item'
        )
        for spec in specs:
            try:
                title = spec.find_element(
                    By.CLASS_NAME, 're__pr-specs-content-item-title'
                ).text.strip()
                value = spec.find_element(
                    By.CLASS_NAME, 're__pr-specs-content-item-value'
                ).text.strip()
                if title in info:       # key phải khớp CHÍNH XÁC với title HTML
                    info[title] = value
            except Exception:
                continue
    except Exception:
        pass

    # ── 4. Mô tả (description) ───────────────────────────────────────────────────
    try:
        info['Mô tả'] = driver.find_element(
            By.CLASS_NAME, 're__section-body.re__detail-content'
        ).text.strip()
    except Exception:
        pass

    return info


In [ ]:
import re

def extract_from_description(desc):
    """
    Trích xuất tối đa thuộc tính từ mô tả rao bán BĐS Việt Nam.
    Trả về dict (NaN nếu không tìm thấy).
    """
    if not isinstance(desc, str) or not desc.strip():
        return {}

    d = desc.lower()
    result = {}

    # ════════════════════════════════════════════════════════════
    # 1. DIỆN TÍCH
    # ════════════════════════════════════════════════════════════
    # Diện tích đất
    m = re.search(r'di[eệ]n t[ií]ch(?:\s+(?:đ[aấ]t|khu[ôô]n đ[aấ]t|thực tế))?[:\s]*(\d+[,.]?\d*)\s*m', d)
    result['DT đất (desc)'] = m.group(1).replace(',', '.') if m else np.nan

    # Diện tích xây dựng / sàn
    m = re.search(r'di[eệ]n t[ií]ch\s*(?:xây dựng|sàn|sd|sử dụng)[:\s]*(\d+[,.]?\d*)\s*m', d)
    result['DT xây dựng (desc)'] = m.group(1).replace(',', '.') if m else np.nan

    # Kích thước thửa đất: 4x20, 5.5x18, (8x18m)
    m = re.search(r'(?:ngang[:\s]*)?\(?(\d+[,.]?\d*)\s*[x×]\s*(\d+[,.]?\d*)\s*m?\)?', d)
    if m:
        result['Ngang (desc)'] = m.group(1).replace(',', '.')
        result['Dài (desc)']   = m.group(2).replace(',', '.')
    else:
        result['Ngang (desc)'] = np.nan
        result['Dài (desc)']   = np.nan

    # Mặt tiền / ngang
    m = re.search(r'(?:m[aặ]t ti[eề]n|ngang)[:\s]*(\d+[,.]?\d*)\s*m', d)
    result['Mặt tiền (desc)'] = m.group(1).replace(',', '.') if m else np.nan

    # ════════════════════════════════════════════════════════════
    # 2. SỐ TẦNG / PHÒNG
    # ════════════════════════════════════════════════════════════
    m = re.search(r'(\d+[,.]?\d*)\s*t[aầ]ng', d)
    result['Số tầng (desc)'] = m.group(1) if m else np.nan

    m = re.search(r'(\d+)\s*(?:phòng ngủ|pn|p\.ngủ|p ngủ|bedroom)', d)
    result['Số PN (desc)'] = m.group(1) if m else np.nan

    m = re.search(r'(\d+)\s*(?:toilet|wc|nhà vệ sinh|phòng tắm|pt|wc)', d)
    result['Số WC (desc)'] = m.group(1) if m else np.nan

    # Tầng hầm / thang máy / sân thượng
    result['Có hầm (desc)']      = 'Có' if re.search(r'(t[aầ]ng h[aầ]m|h[aầ]m xe|basement)', d) else 'Không'
    result['Có thang máy (desc)']= 'Có' if re.search(r'thang m[aá]y|elevator|lift', d) else 'Không'
    result['Có sân thượng (desc)']='Có' if re.search(r's[aâ]n th[ưượ]ng|rooftop|sân thượng', d) else 'Không'

    # ════════════════════════════════════════════════════════════
    # 3. GIÁ
    # ════════════════════════════════════════════════════════════
    m = re.search(r'gi[aá](?:\s+b[aá]n)?[:\s]*(\d+[,.]?\d*)\s*(t[yỷ]|tri[eệ]u|tr\.?|tỉ)', d)
    if m:
        don_vi = 'tỷ' if 't' in m.group(2)[0] else 'triệu'
        result['Giá (desc)'] = m.group(1).replace(',', '.') + ' ' + don_vi
    else:
        result['Giá (desc)'] = np.nan

    # Giá/m²
    m = re.search(r'(\d+[,.]?\d*)\s*(?:tri[eệ]u|tr)\.?\s*/\s*m', d)
    result['Giá/m² (desc)'] = m.group(1).replace(',', '.') + ' tr/m²' if m else np.nan

    # ════════════════════════════════════════════════════════════
    # 4. HƯỚNG NHÀ / BAN CÔNG
    # ════════════════════════════════════════════════════════════
    HUONG = r'(đông[\s-]?nam|đông[\s-]?bắc|tây[\s-]?nam|tây[\s-]?bắc|đông|tây|nam|bắc)'
    m = re.search(r'hướng(?:\s+(?:cửa chính|nhà|chính))?[:\s]*' + HUONG, d)
    result['Hướng nhà (desc)'] = m.group(1).title() if m else np.nan

    m = re.search(r'hướng ban công[:\s]*' + HUONG, d)
    result['Hướng BC (desc)'] = m.group(1).title() if m else np.nan

    # ════════════════════════════════════════════════════════════
    # 5. PHÁP LÝ
    # ════════════════════════════════════════════════════════════
    phap_ly_map = {
        'sổ hồng': 'Sổ hồng', 'sổ đỏ': 'Sổ đỏ',
        'shr': 'Sổ hồng', 'so hong': 'Sổ hồng',
        'hợp đồng mua bán': 'HĐMB', 'hdmb': 'HĐMB',
        'giấy tờ tay': 'Giấy tay', 'giấy tay': 'Giấy tay',
        'chờ sổ': 'Chờ sổ', 'đang làm sổ': 'Chờ sổ',
    }
    result['Pháp lý (desc)'] = np.nan
    for k, v in phap_ly_map.items():
        if k in d:
            result['Pháp lý (desc)'] = v
            break

    # ════════════════════════════════════════════════════════════
    # 6. NỘI THẤT
    # ════════════════════════════════════════════════════════════
    if re.search(r'full n[oộ]i th[aấ]t|đầy đủ n[oộ]i th[aấ]t|n[oộ]i th[aấ]t cao c[aấ]p', d):
        result['Nội thất (desc)'] = 'Đầy đủ'
    elif re.search(r'n[oộ]i th[aấ]t c[oơ] b[aả]n|c[oơ] b[aả]n', d):
        result['Nội thất (desc)'] = 'Cơ bản'
    elif re.search(r'không n[oộ]i th[aấ]t|bàn giao thô|nhà thô', d):
        result['Nội thất (desc)'] = 'Không'
    else:
        result['Nội thất (desc)'] = np.nan

    # ════════════════════════════════════════════════════════════
    # 7. ĐẶC ĐIỂM ĐƯỜNG / HẺM / VỊ TRÍ
    # ════════════════════════════════════════════════════════════
    # Xe hơi / ô tô vào được
    result['Ô tô vào được (desc)'] = 'Có' if re.search(
        r'(hẻm xe (hơi|tải|ô tô)|ô tô ngủ trong|xe tải|xe hơi vào|xe 7 chỗ|oto)', d) else 'Không'

    # Hẻm hay mặt tiền
    if re.search(r'm[aặ]t ti[eề]n|mt\b|mặt phố', d):
        result['Loại mặt đường (desc)'] = 'Mặt tiền'
    elif re.search(r'hxh|hẻm xe hơi', d):
        result['Loại mặt đường (desc)'] = 'Hẻm xe hơi'
    elif re.search(r'hẻm', d):
        result['Loại mặt đường (desc)'] = 'Hẻm'
    else:
        result['Loại mặt đường (desc)'] = np.nan

    # Độ rộng đường vào
    m = re.search(r'(?:đường|hẻm|lộ giới|lg)[:\s]*(\d+[,.]?\d*)\s*m', d)
    result['Rộng đường (desc)'] = m.group(1).replace(',', '.') if m else np.nan

    # ════════════════════════════════════════════════════════════
    # 8. TIỆN ÍCH GẦN KHU VỰC (boolean flags)
    # ════════════════════════════════════════════════════════════
    tien_ich = {
        'Gần chợ':            r'(gần chợ|cạnh chợ|sát chợ|chợ \w{1,20}\s*(\d+m|\d+ m)?)',
        'Gần trường học':     r'(gần trường|cạnh trường|trường học|trường tiểu học|thpt|thcs|mẫu giáo)',
        'Gần bệnh viện':      r'(bệnh viện|bv\b|phòng khám)',
        'Gần siêu thị':       r'(siêu thị|vinmart|coopmart|lotte|bigc|aeon|go!|tops market)',
        'Gần trung tâm TM':   r'(trung tâm thương mại|tttm|vincom|nowzone|gigamall)',
        'Gần công viên':      r'(công viên|cv\b|cây xanh|hồ bơi công cộng)',
        'Gần sông/kênh':      r'(sông|kênh|rạch|bờ sông|view sông|ven sông)',
        'Gần sân bay':        r'(sân bay|tsn|tân sơn nhất)',
        'Gần BX/Ga':          r'(bến xe|nhà ga|metro|tàu điện|ga tàu)',
        'Gần KCN':            r'(khu công nghiệp|kcn\b|khu chế xuất|kcx\b)',
        'Gần ĐH/CĐ':          r'(đại học|cao đẳng|trường đh|trường cđ)',
        'Gần ngân hàng':      r'(ngân hàng|bank|atm)',
        'Gần hồ bơi':         r'(hồ bơi|bể bơi|swimming pool)',
        'Gần gym/TTTD':       r'(gym|phòng gym|thể dục|thể thao|fitness)',
        'Gần nhà thờ/chùa':   r'(nhà thờ|chùa|đình|miếu)',
        'Gần cầu':            r'(cầu \w{1,20}|gần cầu)',
        'Có hồ bơi riêng':    r'(hồ bơi riêng|hồ bơi nội khu|hồ bơi gia đình)',
        'Khu dân cư/dự án':   r'(khu dân cư|kdc|vinhomes|gamuda|nam long|phú mỹ hưng|the manor|sala|eco)',
        'An ninh 24/7':       r'(an ninh|bảo vệ 24|camera an ninh|security|bảo vệ)',
        'Gần biển':           r'(biển|bãi biển|bờ biển|view biển)',
    }
    for label, pattern in tien_ich.items():
        result[label] = 'Có' if re.search(pattern, d) else 'Không'

    # ════════════════════════════════════════════════════════════
    # 9. TỪ KHÓA BÁN / ĐỘNG LỰC BÁN
    # ════════════════════════════════════════════════════════════
    result['Cần bán gấp'] = 'Có' if re.search(
        r'(b[aá]n g[aấ]p|kẹt ti[eề]n|cần ti[eề]n|bán lỗ|bán nhanh|ưu ti[eê]n)', d) else 'Không'
    result['Thương lượng'] = 'Có' if re.search(r'(thương lượng|tl\b|có thể tl|xem xét)', d) else 'Không'
    result['Chính chủ']   = 'Có' if re.search(r'(chính chủ|chủ nhà|chủ bán)', d) else 'Không'

    # ════════════════════════════════════════════════════════════
    # 10. MỤC ĐÍCH SỬ DỤNG PHÙ HỢP
    # ════════════════════════════════════════════════════════════
    result['Phù hợp KD']      = 'Có' if re.search(r'(kinh doanh|buôn bán|cho thuê|mặt bằng kd)', d) else 'Không'
    result['Phù hợp ở']       = 'Có' if re.search(r'(để ở|an cư|vào ở ngay|ở ngay|phù hợp ở)', d) else 'Không'
    result['Phù hợp đầu tư']  = 'Có' if re.search(r'(đầu tư|sinh lời|dòng ti[eề]n|cho thuê|roi\b)', d) else 'Không'

    return result


# ── Áp dụng lên toàn bộ DataFrame ───────────────────────────────────────────
# df = pd.read_csv('data_bds.csv', encoding='utf-8-sig')
# desc_df = pd.DataFrame(df['Mô tả'].apply(extract_from_description).tolist())
# df = pd.concat([df, desc_df], axis=1)
# df.to_csv('data_bds_expanded.csv', index=False, encoding='utf-8-sig')


In [ ]:
def create_driver():
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.page_load_strategy = 'normal'
    return uc.Chrome(options=options)

driver = create_driver()
# Đọc danh sách URLs từ file
with open("hrefs.txt", "r", encoding="utf-8") as file:
    urls = file.read().splitlines()


csv_filename = "data_bds.csv"
total_urls = len(urls)

for index, url in enumerate(urls, start=1):
    try:
        print(f"Đang xử lý link thứ {index}/{total_urls}: {url}")
        property_info = extract_property_info(url)
        
        df = pd.DataFrame([property_info])
        if index == 1:
            df.to_csv(csv_filename, mode='w', header=True, index=False, encoding='utf-8-sig')
        else:
            df.to_csv(csv_filename, mode='a', header=False, index=False, encoding='utf-8-sig')
        
        print(f"Đã lưu thông tin của link thứ {index} vào file CSV")
        time.sleep(0.1)  
    except Exception as e:
        print(f"Lỗi khi xử lý link thứ {index}/{total_urls} - {url}: {str(e)}")

        driver.quit()
        driver = create_driver()

driver.quit()

Đang xử lý link thứ 1/3500: https://batdongsan.com.vn/ban-nha-rieng-duong-doan-gioi-phuong-son-ky-70/ep-48m2-hiem-khu-aeon-tan-phu-hem-xe-tai-7-cho-e-trong-2-mat-hem-thong-pr45718057
Đã lưu thông tin của link thứ 1 vào file CSV
Đang xử lý link thứ 2/3500: https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-duong-nguyen-xien-phuong-long-binh-3-the-manhattan-glory-vinhomes-grand-park/chinh-chu-gui-ban-lo-pho-truc-sinh-23-9-ty-144-m2-so-hong-pr45709145
Đã lưu thông tin của link thứ 2 vào file CSV
Đang xử lý link thứ 3/3500: https://batdongsan.com.vn/ban-nha-rieng-duong-doan-gioi-phuong-son-ky-70/ep-48m2-hiem-khu-aeon-tan-phu-hem-xe-tai-7-cho-e-trong-2-mat-hem-thong-pr45718057
Đã lưu thông tin của link thứ 3 vào file CSV
Đang xử lý link thứ 4/3500: https://batdongsan.com.vn/ban-nha-biet-thu-lien-ke-xa-long-hoa-vinhomes-green-paradise/quy-ngoai-giao-can-gio-chiet-khau-30-ky-truc-tiep-c-t-lh-pr45658571
Đã lưu thông tin của link thứ 4 vào file CSV
Đang xử lý link thứ 5/3500: https://batdongsan